# ParlayAPI quickstart: odds to a pandas DataFrame in one page

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JacobiusMakes/parlayapi-notebooks/blob/main/01-quickstart.ipynb)

This notebook fetches live sportsbook odds from [ParlayAPI](https://parlay-api.com),
flattens the nested JSON into a tidy pandas DataFrame (one row per
event / bookmaker / market / outcome), finds the best available price per outcome,
and saves the snapshot to CSV.

**What you need:** nothing, to taste it. Without a key the notebook uses the keyless
demo endpoint (`/v1/try/...`), which serves the first 5 events of a sport, moneyline
only, capped at 60 requests per hour. A [free API key](https://parlay-api.com/signup) (no card) unlocks
all events, all 30+ sportsbooks, and every market.

In [1]:
# ---- Config: paste your API key between the quotes ----
# Get a free key at https://parlay-api.com/signup (free tier, no card required).
# Leave it empty to run against the keyless demo endpoint instead.
API_KEY = ""

import os
API_KEY = (API_KEY or os.environ.get("PARLAYAPI_KEY", "")).strip()
BASE_URL = "https://parlay-api.com"
SPORT = "baseball_mlb"  # also try: basketball_nba, americanfootball_nfl, icehockey_nhl, soccer_epl

if API_KEY:
    print("API key set: using the full keyed endpoints.")
else:
    print("No API key set: falling back to the keyless demo endpoint.")
    print("The demo serves the first 5 events per sport, moneyline (h2h) only,")
    print("capped at 60 requests per hour. Paste a free key above for all events,")
    print("all 30+ books, and every market:", "https://parlay-api.com/signup")

No API key set: falling back to the keyless demo endpoint.
The demo serves the first 5 events per sport, moneyline (h2h) only,
capped at 60 requests per hour. Paste a free key above for all events,
all 30+ books, and every market: https://parlay-api.com/signup


In [2]:
import requests

def fetch_odds(sport=None, markets="h2h,spreads,totals", odds_format="american"):
    """Fetch current odds as a list of event dicts.

    Keyed:   GET /v1/sports/{sport}/odds returns a bare JSON array of events
             (the-odds-api compatible shape).
    Keyless: GET /v1/try/{sport}/odds returns a demo envelope instead: the
             events are nested under the "events" key, next to demo metadata
             like demo_message and demo_remaining_hour. The two shapes are
             NOT the same at the top level, so we unwrap here.

    Each event: id, home_team, away_team, commence_time, and
    bookmakers[] -> markets[] -> outcomes[] with American prices by default.
    """
    sport = sport or SPORT
    if API_KEY:
        resp = requests.get(
            f"{BASE_URL}/v1/sports/{sport}/odds",
            params={"markets": markets, "oddsFormat": odds_format},
            headers={"X-API-Key": API_KEY},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    resp = requests.get(f"{BASE_URL}/v1/try/{sport}/odds", timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    # Demo envelope: {"demo": true, "demo_message": "...", "events": [...]}
    return payload.get("events", [])

try:
    events = fetch_odds()
except Exception as exc:
    events = []
    print(f"Fetch failed ({exc}). Check your connection or key and re-run this cell.")
print(f"Fetched {len(events)} upcoming {SPORT} events")
if events:
    ev = events[0]
    print("First event:", ev["away_team"], "at", ev["home_team"], "starting", ev["commence_time"])
else:
    print("No events right now (quiet slate or off-season). Re-run later or change SPORT.")

Fetched 5 upcoming baseball_mlb events
First event: Colorado Rockies at Atlanta Braves starting 2026-08-29T20:10:00Z


## Flatten the nested JSON

The odds payload is nested three levels deep: events contain bookmakers, bookmakers
contain markets, markets contain outcomes. For analysis you almost always want it
flat: one row per priced outcome.

In [3]:
import pandas as pd
from datetime import datetime, timezone

def flatten(events):
    fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    rows = []
    for ev in events:
        for bm in ev.get("bookmakers", []):
            for mkt in bm.get("markets", []):
                for out in mkt.get("outcomes", []):
                    rows.append({
                        "fetched_at": fetched_at,
                        "event_id": ev.get("id"),
                        "commence_time": ev.get("commence_time"),
                        "home_team": ev.get("home_team"),
                        "away_team": ev.get("away_team"),
                        "bookmaker": bm.get("key"),
                        "market": mkt.get("key"),
                        "outcome": out.get("name"),
                        "price": out.get("price"),
                        "point": out.get("point"),  # spread / total line, None for h2h
                    })
    return pd.DataFrame(rows)

df = flatten(events)
print(f"{len(df)} priced outcomes across {df['bookmaker'].nunique() if not df.empty else 0} bookmakers")
df.head(10)

134 priced outcomes across 14 bookmakers


,fetched_at,event_id,commence_time,home_team,away_team,bookmaker,market,outcome,price,point
0,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,fanduel,h2h,Atlanta Braves,-220,None
1,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,fanduel,h2h,Colorado Rockies,200,None
2,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,pinnacle,h2h,Atlanta Braves,-212,None
3,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,pinnacle,h2h,Colorado Rockies,192,None
4,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,fliff,h2h,Atlanta Braves,-240,None
5,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,fliff,h2h,Colorado Rockies,190,None
6,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,betrivers,h2h,Atlanta Braves,-225,None
7,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,betrivers,h2h,Colorado Rockies,188,None
8,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,kalshi,h2h,Atlanta Braves,1567,None
9,2026-08-29T13:23:16+00:00,259915ff5c444b16956565f568a05ce2,2026-08-29T20:10:00Z,Atlanta Braves,Colorado Rockies,novig,h2h,Atlanta Braves,-208,None


## Best price per outcome (line shopping, with one honest guard)

American odds compare cleanly as plain numbers: a bigger number always pays more
(+120 beats -110, and -105 beats -110). So the best available price per outcome is
almost just an `idxmax`.

Almost: a thin or mismatched listing (an exchange with no liquidity, a stale
board) can post a price far better than anything a real book will honor, and a
naive max crowns exactly those rows. So first drop any price more than 15% above
the outcome's median decimal price. Production line-shopping boards apply the
same kind of outlier guard.

In [4]:
if df.empty:
    print("Nothing to shop: no rows fetched.")
else:
    h2h = df[df["market"] == "h2h"].copy()
    # American -> decimal so prices compare on a multiplicative scale.
    h2h["decimal"] = [1 + p / 100 if p > 0 else 1 + 100 / (-p) for p in h2h["price"]]
    med = h2h.groupby(["event_id", "outcome"])["decimal"].transform("median")
    shoppable = h2h[h2h["decimal"] <= med * 1.15]
    dropped = len(h2h) - len(shoppable)
    if dropped:
        print(f"outlier guard dropped {dropped} listing(s) priced far off the market median")
    best = shoppable.loc[shoppable.groupby(["event_id", "outcome"])["price"].idxmax(),
                         ["home_team", "away_team", "outcome", "bookmaker", "price"]]
    best = best.sort_values(["home_team", "outcome"]).reset_index(drop=True)
    display(best)

outlier guard dropped 4 listing(s) priced far off the market median


,home_team,away_team,outcome,bookmaker,price
0,Atlanta Braves,Colorado Rockies,Atlanta Braves,novig,-208
1,Atlanta Braves,Colorado Rockies,Colorado Rockies,draftkings,206
2,Chicago Cubs,Cincinnati Reds,Chicago Cubs,fliff,-140
3,Chicago Cubs,Cincinnati Reds,Cincinnati Reds,novig,170
4,Cleveland Guardians,Kansas City Royals,Cleveland Guardians,bet365,-140
5,Cleveland Guardians,Kansas City Royals,Kansas City Royals,novig,141
6,Detroit Tigers,Los Angeles Dodgers,Detroit Tigers,bet365,170
7,Detroit Tigers,Los Angeles Dodgers,Los Angeles Dodgers,novig,-170
8,Milwaukee Brewers,Texas Rangers,Milwaukee Brewers,novig,-153
9,Milwaukee Brewers,Texas Rangers,Texas Rangers,prophetx,150


## Save the snapshot to CSV

One file per run. Collect these on a schedule and you have the raw material for
line-movement and closing-line studies (notebooks 03 and 04 in this series).

In [5]:
csv_path = "parlayapi_odds_snapshot.csv"
if df.empty:
    print("Skipping CSV write: no rows.")
else:
    df.to_csv(csv_path, index=False)
    print(f"Wrote {len(df)} rows to {csv_path}")

Wrote 134 rows to parlayapi_odds_snapshot.csv


---

**More ParlayAPI resources**

- Docs: [parlay-api.com/docs](https://parlay-api.com/docs)
- Free API key (no card): [parlay-api.com/signup](https://parlay-api.com/signup)
- Browser calculators the math here matches: [no-vig](https://parlay-api.com/tools/no-vig-calculator), [parlay](https://parlay-api.com/tools/parlay-calculator), [EV](https://parlay-api.com/tools/ev-calculator)
- The rest of this series: [github.com/JacobiusMakes/parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks)

These notebooks are for research and education. Nothing here is betting advice.